[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheGhoul21/QC-notebooks/blob/main/QC-Intro/03_algorithms_as_interference.ipynb)

# Algoritmi come Interferenza Ingegnerizzata — Deutsch-Jozsa e Grover

**Notebook 3** della serie *QC-Intro: Il Quantum Computing attraverso l'Interferenza*

In questo notebook vedremo come i **veri algoritmi quantistici** non siano altro che applicazioni concrete del paradigma dell'interferenza. Ogni algoritmo segue lo stesso schema:

1. **Preparare** una superposizione uniforme
2. **Marcare** gli stati "buoni" con una fase
3. **Interferire** per convertire le fasi in probabilità
4. **Misurare** per estrarre la risposta

Questo è il **ricettario universale** del quantum computing.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown, Checkbox
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

from qc_intro_utils import (
    apply_style, draw_amplitude_bars, draw_circuit_amplitude_evolution,
    draw_grover_iteration, draw_grover_2d_plane, draw_dj_step_by_step,
    phase_to_color, PRIMARY_COLOR, ACCENT_COLOR, GHOST_COLOR
)

apply_style()
%matplotlib inline

---
## 1. Il template — superposizione → oracolo → interferenza → misura

Ogni algoritmo quantistico che offre un vantaggio rispetto al classico segue lo stesso schema a quattro fasi:

| Fase | Operazione | Effetto sullo stato |
|------|-----------|--------------------|
| **Prepare** | $H^{\otimes n}\lvert 0\rangle^{\otimes n}$ | Superposizione uniforme: tutti i $2^n$ stati con ampiezza $1/\sqrt{2^n}$ |
| **Mark** | Oracolo $U_f$ | Phase kickback: gli stati "buoni" ricevono una fase $-1$ |
| **Interfere** | $H^{\otimes n}$ o diffusore | Le fasi si convertono in differenze di probabilità |
| **Measure** | Misura nella base computazionale | La probabilità è concentrata sulla risposta |

Il **vantaggio quantistico** nasce dal fatto che il passo di interferenza fa un lavoro esponenziale in profondità lineare: $2^n$ ampiezze interferiscono simultaneamente grazie alla linearità della meccanica quantistica.

Algoritmi diversi differiscono nel *come* implementano l'oracolo e il passo di interferenza, ma il template è universale.

In [ ]:
# Static 4-block pipeline diagram
fig, ax = plt.subplots(figsize=(14, 3))
ax.set_xlim(-0.5, 16)
ax.set_ylim(-1, 2.5)
ax.axis('off')

blocks = [
    (0.5, 'PREPARE', r'$H^{\otimes n}$', 'Superposizione\nuniforme', '#4a90d9'),
    (4.5, 'MARK', r'$U_f$', 'Oracolo\n(phase kick)', '#e07b39'),
    (8.5, 'INTERFERE', r'$H^{\otimes n}$', 'Fase → probabilità', '#50a14f'),
    (12.5, 'MEASURE', r'$\langle\psi|$', 'Risposta\nconcentrata', '#c05050'),
]

for x, label, math, desc, color in blocks:
    rect = FancyBboxPatch((x, 0), 3, 1.8, boxstyle='round,pad=0.1',
                          facecolor=color, alpha=0.2, edgecolor=color, lw=2)
    ax.add_patch(rect)
    ax.text(x + 1.5, 1.4, label, ha='center', va='center',
            fontsize=11, fontweight='bold', color=color)
    ax.text(x + 1.5, 0.9, math, ha='center', va='center', fontsize=13)
    ax.text(x + 1.5, 0.3, desc, ha='center', va='center', fontsize=8, color='gray')

# Arrows between blocks
for i in range(3):
    x_start = blocks[i][0] + 3.1
    x_end = blocks[i+1][0] - 0.1
    ax.annotate('', xy=(x_end, 0.9), xytext=(x_start, 0.9),
                arrowprops=dict(arrowstyle='->', color='gray', lw=2))

ax.set_title('Il template universale degli algoritmi quantistici', fontsize=13, pad=10)
plt.tight_layout()
plt.show()

### Forward: Ogni passo è necessario

Vediamo il template in azione su 2 qubit con un oracolo che "marca" lo stato $\lvert 3\rangle = \lvert 11\rangle$ cambiandone la fase. Accendiamo un passo alla volta:

In [ ]:
def template_widget(prepare=False, mark=False, interfere=False):
    """Show the effect of toggling each step of the quantum template."""
    n_qubits = 2
    N = 2 ** n_qubits
    target = 3  # |11>
    
    # Start: |00>
    amps = np.zeros(N, dtype=complex)
    amps[0] = 1.0
    label = 'Stato iniziale |00⟩'
    
    if prepare:
        # Apply H^n: uniform superposition
        amps = np.ones(N, dtype=complex) / np.sqrt(N)
        label = 'Dopo PREPARE'
    
    if mark:
        # Flip phase of target
        amps[target] *= -1
        label = 'Dopo MARK' if not prepare else 'Dopo PREPARE + MARK'
    
    if interfere:
        # Apply H^n (Hadamard transform)
        H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
        Hn = H
        for _ in range(n_qubits - 1):
            Hn = np.kron(Hn, H)
        amps = Hn @ amps
        label = 'Dopo INTERFERE' if not (prepare and mark) else 'Dopo PREPARE + MARK + INTERFERE'
    
    fig, ax = plt.subplots(figsize=(6, 4))
    mags = np.abs(amps)
    phases = np.angle(amps)
    colors = [phase_to_color(p) if m > 1e-10 else (0.85, 0.85, 0.85)
              for m, p in zip(mags, phases)]
    ax.bar(range(N), mags, color=colors, edgecolor='gray', lw=0.5)
    ax.set_xticks(range(N))
    ax.set_xticklabels([f'|{i}⟩' for i in range(N)])
    ax.set_ylabel('|ampiezza|')
    ax.set_ylim(0, 1.15)
    ax.set_title(label, fontsize=12)
    
    # Annotate probabilities
    for i, m in enumerate(mags):
        if m > 1e-10:
            prob = m**2
            ax.text(i, m + 0.03, f'p={prob:.2f}', ha='center', fontsize=8, color='gray')
    
    plt.tight_layout()
    plt.show()

interact(template_widget,
         prepare=Checkbox(value=False, description='Prepare (H⊗n)'),
         mark=Checkbox(value=False, description='Mark (oracolo)'),
         interfere=Checkbox(value=False, description='Interfere (H⊗n)'));

### Backward: Se manca un passo, l'algoritmo fallisce

Provate a:
- Attivare solo **Mark**: la fase cambia, ma le probabilità restano identiche. Senza preparazione, non c'è superposizione su cui l'oracolo possa agire.
- Attivare **Prepare + Mark** senza **Interfere**: vedrete che tutte le probabilità sono ancora $1/4$. La fase c'è, ma senza interferenza non diventa probabilità.
- Attivare tutti e tre: **solo ora** la risposta emerge. Lo stato $\lvert 3\rangle$ viene amplificato.

Questo conferma il principio fondamentale: **l'informazione è nelle fasi, e serve l'interferenza per estrarla**.

---
## 2. Deutsch-Jozsa — il caso più semplice

### Il problema

Data una funzione $f:\{0,1\}\to\{0,1\}$ che è:
- **Costante**: $f(0) = f(1)$ (entrambi 0 oppure entrambi 1)
- **Bilanciata**: $f(0) \neq f(1)$

Determinare se $f$ è costante o bilanciata.

**Classicamente**: servono **2 query** (devi valutare $f(0)$ e $f(1)$).
**Quantisticamente**: basta **1 query** (algoritmo di Deutsch).

### Le 4 funzioni possibili

| Funzione | $f(0)$ | $f(1)$ | Tipo |
|----------|--------|--------|------|
| $f_1$ | 0 | 0 | Costante |
| $f_2$ | 1 | 1 | Costante |
| $f_3$ | 0 | 1 | Bilanciata |
| $f_4$ | 1 | 0 | Bilanciata |

### Il circuito

$$\lvert 0\rangle\lvert 1\rangle \xrightarrow{H^{\otimes 2}} \xrightarrow{U_f} \xrightarrow{H \text{ su q}_0} \text{Misura su } q_0$$

- Se $f$ è costante → misuriamo $\lvert 0\rangle$ con certezza sul primo qubit
- Se $f$ è bilanciata → misuriamo $\lvert 1\rangle$ con certezza sul primo qubit

Il meccanismo è il **phase kickback**: l'oracolo non cambia il qubit di input direttamente, ma la sua *fase* viene "calciata indietro" dal qubit ancilla.

### Forward: Le 4 funzioni passo per passo

Costruiamo il circuito con Qiskit e tracciamo l'evoluzione dello statevector per ciascuna delle 4 funzioni.

In [ ]:
def build_deutsch_circuit(f_type):
    """Build Deutsch circuit for one of the 4 possible 1-bit functions.
    
    f_type: 'constant_0', 'constant_1', 'balanced_id', 'balanced_not'
    Returns: list of (label, statevector_array) stages
    """
    qc = QuantumCircuit(2)
    
    # Prepare: |0>|1>
    qc.x(1)
    sv0 = Statevector.from_instruction(qc)
    
    # Hadamard both
    qc.h([0, 1])
    sv1 = Statevector.from_instruction(qc)
    
    # Oracle
    if f_type == 'constant_0':
        pass  # identity
    elif f_type == 'constant_1':
        qc.x(1)  # flip ancilla
    elif f_type == 'balanced_id':
        qc.cx(0, 1)  # f(x) = x
    elif f_type == 'balanced_not':
        qc.x(0)
        qc.cx(0, 1)
        qc.x(0)  # f(x) = 1-x
    sv2 = Statevector.from_instruction(qc)
    
    # Final Hadamard on qubit 0
    qc.h(0)
    sv3 = Statevector.from_instruction(qc)
    
    stages = [
        ('|0⟩|1⟩', sv0.data),
        ('Dopo H⊗2', sv1.data),
        (f'Dopo oracolo', sv2.data),
        ('Dopo H su q₀', sv3.data),
    ]
    return stages

# Show all 4 functions
f_names = {
    'constant_0': 'f₁: f(x)=0  (costante)',
    'constant_1': 'f₂: f(x)=1  (costante)',
    'balanced_id': 'f₃: f(x)=x  (bilanciata)',
    'balanced_not': 'f₄: f(x)=1−x (bilanciata)',
}

for f_type, name in f_names.items():
    stages = build_deutsch_circuit(f_type)
    fig, axes = draw_circuit_amplitude_evolution(stages)
    fig.suptitle(name, fontsize=13, y=1.05)
    plt.show()

### Backward: Dalla misura alla funzione

Osservate l'ultimo pannello di ogni riga:

- **Funzioni costanti** ($f_1, f_2$): l'ampiezza è concentrata sugli stati con il primo qubit = $\lvert 0\rangle$ (cioè $\lvert 00\rangle$ e $\lvert 01\rangle$). Se misuriamo il primo qubit, otteniamo **sempre** $0$.
- **Funzioni bilanciate** ($f_3, f_4$): l'ampiezza è concentrata sugli stati con il primo qubit = $\lvert 1\rangle$ (cioè $\lvert 10\rangle$ e $\lvert 11\rangle$). Se misuriamo il primo qubit, otteniamo **sempre** $1$.

Quindi:
$$\text{Misuro } \lvert 0\rangle \Rightarrow f \text{ è costante}, \qquad \text{Misuro } \lvert 1\rangle \Rightarrow f \text{ è bilanciata}$$

**Una sola query, risposta certa.** Il classico ne serve due.

### Explore: Seleziona la funzione

In [ ]:
def dj_1qubit_widget(funzione='f₁: f(x)=0 (costante)'):
    """Interactive exploration of Deutsch's algorithm."""
    f_map = {
        'f₁: f(x)=0 (costante)': 'constant_0',
        'f₂: f(x)=1 (costante)': 'constant_1',
        'f₃: f(x)=x (bilanciata)': 'balanced_id',
        'f₄: f(x)=1−x (bilanciata)': 'balanced_not',
    }
    f_type = f_map[funzione]
    stages = build_deutsch_circuit(f_type)
    fig, axes = draw_circuit_amplitude_evolution(stages)
    fig.suptitle(funzione, fontsize=13, y=1.05)
    
    # Annotate the result
    sv_final = np.array(stages[-1][1])
    # Probability of measuring |0> on first qubit: sum |<0x|psi>|^2 for x in {0,1}
    p0 = abs(sv_final[0])**2 + abs(sv_final[1])**2
    p1 = abs(sv_final[2])**2 + abs(sv_final[3])**2
    result = 'COSTANTE' if p0 > 0.99 else 'BILANCIATA'
    q0_val = '0' if p0 > 0.99 else '1'
    axes[-1].text(0.5, -0.15, f'Misura q₀ = |{q0_val}⟩ → {result}',
                  transform=axes[-1].transAxes, ha='center', fontsize=10,
                  fontweight='bold', color=ACCENT_COLOR)
    plt.show()

interact(dj_1qubit_widget,
         funzione=Dropdown(options=[
             'f₁: f(x)=0 (costante)',
             'f₂: f(x)=1 (costante)',
             'f₃: f(x)=x (bilanciata)',
             'f₄: f(x)=1−x (bilanciata)',
         ], description='Funzione:'));

---
## 3. Deutsch-Jozsa generalizzato — n qubit

### Il problema scalato

Data una funzione $f:\{0,1\}^n \to \{0,1\}$ che è:
- **Costante**: $f(x) = c$ per ogni $x$ (stessa uscita su tutti i $2^n$ input)
- **Bilanciata**: esattamente metà degli input dà 0, metà dà 1

Determinare se $f$ è costante o bilanciata.

**Classicamente**: nel caso peggiore servono $2^{n-1}+1$ query (bisogna verificare più di metà degli input).
**Quantisticamente**: basta **1 sola query**, indipendentemente da $n$.

### Il circuito

$$\lvert 0\rangle^{\otimes n}\lvert 1\rangle \xrightarrow{H^{\otimes (n+1)}} \xrightarrow{U_f} \xrightarrow{H^{\otimes n}} \text{Misura sui primi } n \text{ qubit}$$

- Se $f$ è costante → tutti zeri sui qubit di input
- Se $f$ è bilanciata → almeno un bit diverso da zero

Al crescere di $n$, il costo classico **esplode** ($2^{n-1}+1$), mentre quello quantistico **resta a 1 query**. Questo è il vantaggio esponenziale di Deutsch-Jozsa.

### Forward: Esempio a 3 qubit

In [ ]:
# 3-qubit Deutsch-Jozsa: constant_0, constant_1, balanced
for f_type in ['constant_0', 'constant_1', 'balanced']:
    fig, axes, stages = draw_dj_step_by_step(f_type, n_qubits=3)
    fig.suptitle(f'DJ a 3 qubit — oracolo: {f_type}', fontsize=13, y=1.05)
    plt.show()

Notate come:
- Per gli oracoli **costanti**, dopo l'Hadamard finale tutta l'ampiezza torna su stati con i primi 3 qubit a $\lvert 000\rangle$.
- Per l'oracolo **bilanciato**, l'ampiezza si ridistribuisce su stati con i primi 3 qubit $\neq \lvert 000\rangle$.

Una misura sui primi 3 qubit distingue i due casi con certezza.

### Explore: Scala il numero di qubit

In [ ]:
def dj_nqubit_widget(n_qubits=2, oracle='constant_0'):
    """Interactive n-qubit Deutsch-Jozsa."""
    fig, axes, stages = draw_dj_step_by_step(oracle, n_qubits=n_qubits)
    fig.suptitle(f'Deutsch-Jozsa: {n_qubits} qubit, oracolo={oracle}', fontsize=13, y=1.05)
    
    # Show classical cost comparison
    N = 2 ** n_qubits
    classical_cost = N // 2 + 1
    axes[-1].text(0.5, -0.15,
                  f'Query quantistiche: 1 | Query classiche (worst case): {classical_cost}',
                  transform=axes[-1].transAxes, ha='center', fontsize=9,
                  fontweight='bold', color=ACCENT_COLOR)
    plt.show()

interact(dj_nqubit_widget,
         n_qubits=IntSlider(min=2, max=5, value=3, description='n qubit:'),
         oracle=Dropdown(options=['constant_0', 'constant_1', 'balanced'],
                         description='Oracolo:'));

---
## 4. Grover — amplificazione iterativa

### Il problema della ricerca

Dato un insieme di $N = 2^n$ elementi e un oracolo che "marca" un elemento target, trovare il target.

- **Classicamente**: $O(N)$ query (forza bruta)
- **Quantisticamente**: $O(\sqrt{N})$ query (Grover)

### L'algoritmo di Grover

1. **Preparazione**: superposizione uniforme $\lvert \psi_0\rangle = H^{\otimes n}\lvert 0\rangle^{\otimes n}$, tutte le ampiezze $= 1/\sqrt{N}$
2. **Iterare** circa $\frac{\pi}{4}\sqrt{N}$ volte:
   - **Oracolo**: cambia segno all'ampiezza del target ($a_{\text{target}} \mapsto -a_{\text{target}}$)
   - **Diffusore**: riflette tutte le ampiezze rispetto alla media ($a_i \mapsto 2\bar{a} - a_i$)
3. **Misura**: il target ha probabilità vicina a 1

### Come funziona il diffusore

Il diffusore è uno specchio che riflette le ampiezze rispetto alla media. L'oracolo abbassa l'ampiezza del target sotto la media. Poi lo specchio la riflette *sopra* la media. Ogni iterazione **pompa** più ampiezza nel target.

$$D = 2\lvert \psi_0\rangle\langle \psi_0\rvert - I$$

### ⚠️ Attenzione: il sovra-iterare è dannoso

Grover è una **rotazione**, non un processo monotono. Se iterate troppo, l'ampiezza del target supera il massimo e torna a scendere. È come una giostra: dovete fermarvi al momento giusto.

### Forward: 3 qubit (8 stati), target = |5⟩

In [ ]:
# Show Grover iterations 0 through 5 on 3 qubits, target=5
# draw_grover_iteration shows the full history up to n_iters
fig, axes, history = draw_grover_iteration(n_qubits=3, target=5, n_iters=5)
fig.suptitle('Grover: 3 qubit, target=|5⟩ — iterazioni da 0 a 5', fontsize=12, y=1.05)
plt.show()

### Backward: Quante iterazioni servono?

Per $N = 2^3 = 8$ stati, il numero ottimale di iterazioni è:

$$k_{\text{opt}} \approx \frac{\pi}{4}\sqrt{N} = \frac{\pi}{4}\sqrt{8} \approx 2.22$$

Quindi **2 iterazioni** danno la probabilità massima. Osservate nei grafici sopra:
- A 0 iterazioni: tutte le ampiezze sono uguali ($1/\sqrt{8} \approx 0.354$)
- A 2 iterazioni: il target $\lvert 5\rangle$ domina
- A 4-5 iterazioni: il target scende di nuovo — abbiamo **sovra-iterato**

**Se voglio una probabilità vicina a 1, devo iterare circa $\pi/4 \cdot \sqrt{N}$ volte. Né più, né meno.**

### Explore 1: Varia qubit e iterazioni

In [ ]:
def grover_widget_1(n_qubits=3, n_iterations=2):
    """Interactive Grover: vary qubits and iterations."""
    N = 2 ** n_qubits
    target = 0  # default target
    
    # Compute amplitudes
    amps = np.ones(N, dtype=complex) / np.sqrt(N)
    for _ in range(n_iterations):
        amps[target] *= -1
        mean = amps.mean()
        amps = 2 * mean - amps
    
    fig, ax = plt.subplots(figsize=(max(N * 0.5, 6), 4))
    mags = np.abs(amps)
    colors = [ACCENT_COLOR if j == target else PRIMARY_COLOR for j in range(N)]
    ax.bar(range(N), mags, color=colors, edgecolor='gray', lw=0.5)
    ax.axhline(1 / np.sqrt(N), color='gray', ls='--', lw=0.5, label=f'1/√{N}')
    ax.set_xticks(range(N))
    ax.set_xticklabels([f'|{j}⟩' for j in range(N)], fontsize=7, rotation=45)
    ax.set_ylabel('|ampiezza|')
    ax.set_ylim(0, 1.15)
    
    # Show optimal iteration count
    k_opt = np.pi / 4 * np.sqrt(N)
    p_target = float(np.abs(amps[target])**2)
    ax.set_title(f'{n_qubits} qubit (N={N}), {n_iterations} iter | '
                 f'k_opt ≈ {k_opt:.1f} | P(target) = {p_target:.3f}',
                 fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

interact(grover_widget_1,
         n_qubits=IntSlider(min=2, max=5, value=3, description='n qubit:'),
         n_iterations=IntSlider(min=0, max=15, value=2, description='Iterazioni:'));

### Explore 2: Scegli il target

In [ ]:
def grover_widget_2(target=5, n_iterations=2):
    """Interactive Grover for 3 qubits: choose target and iterations."""
    n_qubits = 3
    N = 2 ** n_qubits
    
    amps = np.ones(N, dtype=complex) / np.sqrt(N)
    for _ in range(n_iterations):
        amps[target] *= -1
        mean = amps.mean()
        amps = 2 * mean - amps
    
    fig, ax = plt.subplots(figsize=(8, 4))
    mags = np.abs(amps)
    colors = [ACCENT_COLOR if j == target else PRIMARY_COLOR for j in range(N)]
    ax.bar(range(N), mags, color=colors, edgecolor='gray', lw=0.5)
    ax.axhline(1 / np.sqrt(N), color='gray', ls='--', lw=0.5, label=f'1/√{N}')
    ax.set_xticks(range(N))
    ax.set_xticklabels([f'|{j}⟩' for j in range(N)], fontsize=9)
    ax.set_ylabel('|ampiezza|')
    ax.set_ylim(0, 1.15)
    
    p_target = float(np.abs(amps[target])**2)
    ax.set_title(f'Grover 3 qubit — target=|{target}⟩, {n_iterations} iter | '
                 f'P(target) = {p_target:.3f}', fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

interact(grover_widget_2,
         target=IntSlider(min=0, max=7, value=5, description='Target:'),
         n_iterations=IntSlider(min=0, max=15, value=2, description='Iterazioni:'));

---
## 5. Grover geometricamente

### L'interpretazione nel piano 2D

Grover ha una bellissima interpretazione geometrica. Definiamo due stati ortogonali:

$$\lvert \text{target}\rangle \qquad \text{e} \qquad \lvert \text{rest}\rangle = \frac{1}{\sqrt{N-1}} \sum_{x \neq \text{target}} \lvert x\rangle$$

Lo stato iniziale $\lvert \psi_0\rangle = H^{\otimes n}\lvert 0\rangle^{\otimes n}$ giace quasi interamente lungo $\lvert \text{rest}\rangle$: ha solo una componente $1/\sqrt{N}$ lungo $\lvert \text{target}\rangle$.

### La rotazione

Ogni iterazione di Grover è una **rotazione di angolo** $2\theta_0$ nel piano 2D $\{\lvert\text{target}\rangle, \lvert\text{rest}\rangle\}$, dove:

$$\theta_0 = \arcsin\left(\frac{1}{\sqrt{N}}\right)$$

Dopo $k$ iterazioni, lo stato ha ruotato di $(2k+1)\theta_0$ dall'asse $\lvert\text{rest}\rangle$.

### Il numero ottimale di iterazioni

Vogliamo che l'angolo totale sia $\pi/2$ (stato completamente allineato con $\lvert\text{target}\rangle$):

$$(2k+1)\theta_0 \approx \frac{\pi}{2} \quad \Rightarrow \quad k \approx \frac{\pi}{4\theta_0} \approx \frac{\pi}{4}\sqrt{N}$$

Questo spiega **tutto**:
- Perché servono $\sqrt{N}$ passi (l'angolo $\theta_0 \sim 1/\sqrt{N}$)
- Perché il sovra-iterare è dannoso (si ruota oltre $\pi/2$)
- Perché più iterazioni peggiorano le cose (rotazione completa e ritorno)

### Forward: Rotazione nel piano per 3 qubit

In [ ]:
# Grover 2D plane for 3 qubits
fig, ax = plt.subplots(figsize=(6, 6))
draw_grover_2d_plane(n_qubits=3, target=5, n_iters=5, ax=ax)
plt.tight_layout()
plt.show()

### Forward 2: Confronto al variare di n

Al crescere di $n$, l'angolo $\theta_0$ si restringe e servono più passi. Ma la geometria è identica.

In [ ]:
# Compare Grover 2D plane for different n
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, n in zip(axes, [3, 5, 8]):
    N = 2 ** n
    k_opt = int(round(np.pi / 4 * np.sqrt(N)))
    draw_grover_2d_plane(n_qubits=n, target=0, n_iters=min(k_opt + 2, 20), ax=ax)
plt.tight_layout()
plt.show()

### Explore: Rotazione interattiva

In [ ]:
def grover_2d_widget(n_qubits=3, n_iterations=5):
    """Interactive 2D Grover rotation plane."""
    N = 2 ** n_qubits
    theta_0 = np.arcsin(1 / np.sqrt(N))
    k_opt = np.pi / (4 * theta_0) - 0.5
    
    fig, ax = plt.subplots(figsize=(6, 6))
    draw_grover_2d_plane(n_qubits=n_qubits, target=0, n_iters=n_iterations, ax=ax)
    
    # Show optimal info
    total_angle = (2 * n_iterations + 1) * theta_0
    p_target = np.sin(total_angle)**2
    ax.text(0.02, 0.98,
            f'θ₀ = {theta_0:.4f}\n'
            f'k_opt ≈ {k_opt:.1f}\n'
            f'angolo totale = {total_angle:.3f} (π/2 = {np.pi/2:.3f})\n'
            f'P(target) = {p_target:.4f}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    plt.tight_layout()
    plt.show()

interact(grover_2d_widget,
         n_qubits=IntSlider(min=2, max=8, value=3, description='n qubit:'),
         n_iterations=IntSlider(min=0, max=30, value=5, description='Iterazioni:'));

---
## 6. Messaggio finale — il quantum computing è scultura di fasi

| | Classico | Quantistico |
|---|---|---|
| **Rappresentazione** | Singolo stato | Superposizione di $2^n$ stati |
| **Informazione** | Nel valore | Nelle fasi relative |
| **Calcolo** | Trasformazione sequenziale | Interferenza ingegnerizzata |
| **Ricerca** | Forza bruta: $O(N)$ | Grover: $O(\sqrt{N})$ |
| **Oracolo** | Chiama e leggi il risultato | Marca con fase, lascia che l'interferenza amplifichi |

### Cosa abbiamo imparato in questa serie

Il quantum computing **NON** è "provare tutto in parallelo". È un processo in quattro atti:

1. **Preparare una superposizione** (Notebook 0: stati e ampiezze)
2. **Codificare informazione nelle FASI** (Notebook 1: interferenza)
3. **Usare gate per creare le condizioni di interferenza giuste** (Notebook 2: gate e circuiti)
4. **Lasciare che la FISICA faccia il filtraggio** (questo notebook: algoritmi)

Ogni algoritmo quantistico — da Deutsch-Jozsa a Shor — è una variazione su questo tema. L'arte sta nel progettare l'oracolo e il passo di interferenza in modo che le ampiezze dei risultati "sbagliati" si cancellino e quelle del risultato "giusto" si sommino costruttivamente.

### La prossima tappa

Il **Quantum Fourier Transform** (la serie *Visual QFT*) è il motore di interferenza più potente in circolazione. È il cuore dell'algoritmo di Shor per la fattorizzazione, del phase estimation, e di gran parte degli algoritmi quantistici più importanti. Là vedremo come l'interferenza viene orchestrata in modo ancora più sofisticato.